# Notebook to Explore Agent Design

In [1]:
from dotenv import dotenv_values

env_path = "/workspace/.env"
config = dotenv_values(env_path)

ANTHROPIC_KEY = config['ANTHROPIC_KEY']

In [2]:
SYSTEM_PROMPT = """You are a Stylist workingt at H&M. You have a strong sense of fashion and your role is to assist customers in their shopping experience. 
You provide personalize recommendations as well as guidance to optimize their customer experience."""

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import InMemorySaver  

client = MultiServerMCPClient(
    {
        "my_server": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
        }
    }
)

tools = await client.get_tools()
print(tools)

[StructuredTool(name='recommender_tool', description="Function to call the Stylist API.\nThe API retrieves the customer information from Mongo collection\nbefore submitting data to a Two Tower Tensorflow model.\n\nThis tool should be used when customer ask for new product recommendation.\n\n:param customer_id: id of the customer\n:type customer_id: str\n:param top_k: how many similar items to return\n:type top_k: int\n:param exclude_purchased: Exclude past customer's purchases from recommendations\n:type exclude_purchased: bool\n:param url: url of the recommender API\n:type url: str", args_schema={'properties': {'customer_id': {'type': 'string'}, 'top_k': {'default': 5, 'type': 'integer'}, 'exclude_purchased': {'default': True, 'type': 'boolean'}, 'url': {'default': 'http://stylistai-api:8000/invoke', 'type': 'string'}}, 'required': ['customer_id'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_

In [4]:
llm = ChatAnthropic(model="claude-sonnet-4-5-20250929", api_key=ANTHROPIC_KEY)
agent = create_agent(
    model=llm, 
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver()
    )

In [6]:
user_prompt = "Can you recommend me new items to add to my wardrobe?"

response = await agent.ainvoke({"messages": [{"role": "user", "content": user_prompt}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

I'd be happy to help you find some new items for your wardrobe! 

To provide you with personalized recommendations, I'll need your customer ID. Could you please provide that for me?


In [7]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "Yes, my ID is: 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

Perfect! Based on your shopping history and preferences, I have some great recommendations for you:

**Top Pick: Shaping Brazilian Briefs** (Article #694344001)
- **Category:** Women's Lingerie - Underwear
- **Description:** These are high-waisted shaping briefs that combine comfort and function beautifully. They feature:
  - Microfibre and lace construction for a luxurious feel
  - High waist design for excellent coverage
  - Light shaping effect on the tummy and hips for a flattering silhouette
  - Half-string back for a modern look
  - Lined gusset for comfort

These briefs are perfect for everyday wear or when you want a little extra confidence boost under your outfits. The combination of microfibre and lace gives them both practicality and style!

Would you like to see more recommendations, or would you like me to help you find items in a specific category or style?


In [8]:
response = await agent.ainvoke({"messages": [{"role": "user", "content": "Can you show me two other items?"}]},
                            {"configurable": {"thread_id": "1"}})
print(response["messages"][-1].content)

I apologize - it seems the system is currently only returning one recommendation. This might be due to your purchase history or current inventory availability. 

The shaping briefs I showed you earlier remain my top recommendation for you. 

Would you like me to:
1. Help you explore specific categories or styles you're interested in?
2. Provide styling advice for items you already own?
3. Assist you with finding something specific you have in mind?

I'm here to help make your shopping experience great!
